# **Kalshi MLB Player Prop Market Analysis**

In [82]:
import numpy as np
import pandas as pd
from utils import *
import plotly.express as px
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing

In [83]:
SERIES_TICKERS = {
    "homeruns"         : "KXMLBHR",    # MLB: 'Buster Posey records n+ home runs in a game'
    "strikeouts"       : "KXMLBKS",    # MLB: 'Tim Lincecum records n+ strike outs in a game'
    "hits"             : "KXMLBHIT",   # MLB: 'Buster Posey records n+ hits in a game'
    "hits_runs_rbi"    : "KXMLBHRR",   # MLB: 'Buster Posey records n+ combined hits, runs, or RBIs in a game'
    "total_bases"      : "KXMLBTB",    # MLB: 'Buster Posey records n+ total bases in a game'
    "outs"             : "KXMLBOUTS",  # MLB: 'Tim Lincecum records n+ total outs'
    "rbi"              : "KXMLBRBI",   # MLB: 'Buster Posey records n+ total RBIs in a game'
    "stolen_bases"     : "KXMLBSB",    # MLB: 'Buster Posey records n+ stolen bases in a game'
}

In [84]:
MONTH_MAP = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12,
}

### Preprocessing - Market Data

In [85]:
# Load in data

recent_markets_df = pd.read_parquet('data/raw/recent_markets.parquet')
historical_markets_df = pd.read_parquet('data/raw/historical_markets.parquet')

In [86]:
recent_markets_df.head(2)

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,...,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,subtitle
0,True,2026-07-11T06:00:26Z,2026-07-10T22:34:20.107856Z,{'baseball_player': 'b160dd99-cd2c-4348-ac4e-4...,This market will close and expire after the ev...,KXMLBHR-26JUL102215COLSF,2026-07-11T05:15:00Z,2026-07-14T02:15:00Z,Willy Adames: 0 home runs,0.5,...,Willy Adames: 1+ home runs?,2026-07-11T06:06:29.51999Z,12001.48,12001.48,1.0000,0.00,0.0000,0.00,Willy Adames: 1+,None
1,True,2026-07-11T06:00:25Z,2026-07-10T22:34:14.897775Z,{'baseball_player': 'c26b6723-732e-48ba-8e2a-4...,This market will close and expire after the ev...,KXMLBHR-26JUL102215COLSF,2026-07-11T05:15:00Z,2026-07-14T02:15:00Z,Willi Castro: 0 home runs,1.5,...,Willi Castro: 2+ home runs?,2026-07-11T06:06:29.51999Z,23.39,23.39,1.0000,0.00,0.0000,0.00,Willi Castro: 2+,None


In [87]:
historical_markets_df.head(2)

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2026-05-11T23:52:57Z,2026-05-11T20:23:23.775122Z,{'baseball_player': 'e5aede12-a1f0-4820-9402-3...,This market will close and expire after the ev...,KXMLBHR-26MAY111835NYYBAL,2026-05-12T01:35:00Z,2026-05-14T22:35:00Z,Cancelled,1.5,...,KXMLBHR-26MAY111835NYYBAL-BALSBASALLO29-2,Samuel Basallo: 2+ home runs?,2026-05-11T23:59:05.204048Z,0.00,0.00,0.0900,-1.00,0.0000,0.00,Samuel Basallo: 2+
1,True,2026-05-11T23:52:57Z,2026-05-11T20:23:18.567172Z,{'baseball_player': 'e5aede12-a1f0-4820-9402-3...,This market will close and expire after the ev...,KXMLBHR-26MAY111835NYYBAL,2026-05-12T01:35:00Z,2026-05-14T22:35:00Z,Cancelled,0.5,...,KXMLBHR-26MAY111835NYYBAL-BALSBASALLO29-1,Samuel Basallo: 1+ home runs?,2026-05-11T23:59:05.204048Z,0.00,9644.13,0.1300,-117.00,0.0000,0.00,Samuel Basallo: 1+


In [88]:
# Join data frames

markets_df = pd.concat([recent_markets_df, historical_markets_df], axis=0, join='outer')

In [89]:
# Drop duplicate tickers

markets_df = markets_df.drop_duplicates(['ticker'])

In [90]:
# Filter for liquid markets

markets_df = markets_df[markets_df['volume_fp'].astype(float) > 0]

In [91]:
# Market is finalized

markets_df = markets_df[markets_df['status'] == 'finalized']

In [92]:
# Remove 'scalar' (ie the game was cancelled)

markets_df = markets_df[markets_df['result'] != 'scalar']

In [93]:
# Clean up index

markets_df = markets_df.reset_index(drop=True)

In [94]:
# Create player column

split_col = markets_df['yes_sub_title'].str.split(':', n=1, expand=True)
markets_df['player']  = split_col[0].str.strip()

markets_df['player'].value_counts()[:10]

player
Shohei Ohtani          1606
Yordan Alvarez         1413
Freddie Freeman        1351
Kyle Schwarber         1344
Matt Olson             1340
Bryce Harper           1336
Nick Kurtz             1326
Pete Crow-Armstrong    1309
Gunnar Henderson       1287
James Wood             1286
Name: count, dtype: int64

In [95]:
# Create series column

markets_df['series'] = markets_df['ticker'].apply(lambda x: x.split('-')[0])

### Preprocessing - Trade Data

In [96]:
# Load in data

recent_trades_df = pd.read_parquet('data/raw/recent_trades.parquet')
historical_trades_df = pd.read_parquet('data/raw/historical_trades.parquet')

In [97]:
recent_trades_df.head(2)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars
0,93.51,2026-07-11T04:47:00.715783Z,False,0.9900,bid,yes,yes,KXMLBHR-26JUL102215COLSF-COLHGOODMAN15-2,aabfb046-ca5d-5d40-e6b7-67f813050a64,0.0100
1,62.42,2026-07-11T02:22:09.932424Z,False,0.9700,bid,yes,yes,KXMLBHR-26JUL102215COLSF-COLHGOODMAN15-2,6c3884bb-8305-7e86-8476-a7a0fe5b4eb5,0.0300


In [98]:
recent_trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2799802 entries, 0 to 2799801
Data columns (total 10 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   count_fp            object
 1   created_time        object
 2   is_block_trade      bool  
 3   no_price_dollars    object
 4   taker_book_side     object
 5   taker_outcome_side  object
 6   taker_side          object
 7   ticker              object
 8   trade_id            object
 9   yes_price_dollars   object
dtypes: bool(1), object(9)
memory usage: 194.9+ MB


In [99]:
historical_trades_df.head(2)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars
0,93.51,2026-05-10T22:07:50.435248Z,False,0.9900,bid,yes,yes,KXMLBHR-26MAY101920DETKC-DETMVIERLING8-2,f2a892a3-30f1-589c-413a-124a61d75ca0,0.0100
1,93.51,2026-05-10T22:12:44.497303Z,False,0.9900,bid,yes,yes,KXMLBHR-26MAY101920DETKC-KCMGARCIA11-2,7b23f626-12c0-5570-f8d7-8ed00c8e1751,0.0100


In [100]:
historical_trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1463001 entries, 0 to 1463000
Data columns (total 10 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    ----- 
 0   count_fp            1463001 non-null  object
 1   created_time        1463001 non-null  object
 2   is_block_trade      1463001 non-null  bool  
 3   no_price_dollars    1463001 non-null  object
 4   taker_book_side     1463001 non-null  object
 5   taker_outcome_side  1463001 non-null  object
 6   taker_side          1463001 non-null  object
 7   ticker              1463001 non-null  object
 8   trade_id            1463001 non-null  object
 9   yes_price_dollars   1463001 non-null  object
dtypes: bool(1), object(9)
memory usage: 101.9+ MB


In [101]:
# Join data frames

trades_df = pd.concat([recent_trades_df, historical_trades_df], axis=0, join='outer')

In [102]:
# Convert data

numeric_cols = ['count_fp', 'yes_price_dollars', 'no_price_dollars']

trades_df[numeric_cols] = trades_df[numeric_cols].apply(pd.to_numeric)

trades_df['created_time'] = pd.to_datetime(trades_df['created_time'], format='ISO8601', utc=True).dt.tz_convert('America/New_York')

In [103]:
# Drop duplicate trades

trades_df = trades_df.drop_duplicates(['trade_id']).reset_index(drop=True)

In [104]:
# Add game start time column

parts = trades_df['ticker'].str.split('-').str[1]

year   = 2000 + parts.str[:2].astype(int)
month  = parts.str[2:5].str.upper().map(MONTH_MAP)
day    = parts.str[5:7].astype(int)
hour   = parts.str[7:9].astype(int)
minute = parts.str[9:11].astype(int)

trades_df['game_start_time'] = pd.to_datetime(
    dict(year=year, month=month, day=day, hour=hour, minute=minute)
).dt.tz_localize('America/New_York')

In [105]:
# Add dollar amount column

trades_df['dollar_amt'] = trades_df['count_fp'] * np.where(
    trades_df['taker_outcome_side'] == 'no', 
    trades_df['no_price_dollars'],
    trades_df['yes_price_dollars']
)

### Preprocessing - Combine Data

In [106]:
markets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256196 entries, 0 to 256195
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   can_close_early           256196 non-null  bool   
 1   close_time                256196 non-null  object 
 2   created_time              256196 non-null  object 
 3   custom_strike             256196 non-null  object 
 4   early_close_condition     256196 non-null  object 
 5   event_ticker              256196 non-null  object 
 6   expected_expiration_time  256196 non-null  object 
 7   expiration_time           256196 non-null  object 
 8   expiration_value          256196 non-null  object 
 9   floor_strike              256196 non-null  float64
 10  last_price_dollars        256196 non-null  object 
 11  latest_expiration_time    256196 non-null  object 
 12  liquidity_dollars         256196 non-null  object 
 13  market_type               256196 non-null  o

In [107]:
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4093557 entries, 0 to 4093556
Data columns (total 12 columns):
 #   Column              Dtype                           
---  ------              -----                           
 0   count_fp            float64                         
 1   created_time        datetime64[ns, America/New_York]
 2   is_block_trade      bool                            
 3   no_price_dollars    float64                         
 4   taker_book_side     object                          
 5   taker_outcome_side  object                          
 6   taker_side          object                          
 7   ticker              object                          
 8   trade_id            object                          
 9   yes_price_dollars   float64                         
 10  game_start_time     datetime64[ns, America/New_York]
 11  dollar_amt          float64                         
dtypes: bool(1), datetime64[ns, America/New_York](2), float64(4), object(5)

In [108]:
# Merge data frames

merged = trades_df.merge(markets_df, on='ticker', how='left')

final = merged[[
    'ticker', 'series', 'created_time_x', 'count_fp', 'close_time', 
    'game_start_time', 'dollar_amt', 'player', 'result', 'yes_price_dollars',
]].rename(columns={'created_time_x': 'created_time'}).dropna().reset_index(drop=True).copy()

final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4055494 entries, 0 to 4055493
Data columns (total 10 columns):
 #   Column             Dtype                           
---  ------             -----                           
 0   ticker             object                          
 1   series             object                          
 2   created_time       datetime64[ns, America/New_York]
 3   count_fp           float64                         
 4   close_time         object                          
 5   game_start_time    datetime64[ns, America/New_York]
 6   dollar_amt         float64                         
 7   player             object                          
 8   result             object                          
 9   yes_price_dollars  float64                         
dtypes: datetime64[ns, America/New_York](2), float64(3), object(5)
memory usage: 309.4+ MB


In [109]:
# Create 'prop' data frames

prop_dfs = {
    name: final[final['series'] == ticker].reset_index(drop=True).copy()
    for name, ticker in SERIES_TICKERS.items()
}

homeruns_df       = prop_dfs['homeruns']       
strikeouts_df     = prop_dfs['strikeouts']
hits_df           = prop_dfs['hits']            
hits_runs_rbi_df  = prop_dfs['hits_runs_rbi']
total_bases_df    = prop_dfs['total_bases']
outs_df           = prop_dfs['outs']
rbi_df            = prop_dfs['rbi']
stolen_bases_df   = prop_dfs['stolen_bases']

In [110]:
print('Dollar Volume by Series:')
final.groupby('series')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Dollar Volume by Series:


series
KXMLBHIT     $14,200,964
KXMLBHR      $28,070,125
KXMLBHRR     $10,184,421
KXMLBKS      $32,366,230
KXMLBOUTS       $542,697
KXMLBRBI        $131,670
KXMLBSB         $126,581
KXMLBTB       $6,715,897
Name: dollar_amt, dtype: object

In [111]:
print('Number of Markets (tickers) by Series:')
final.groupby('series')['ticker'].count().apply(lambda x: f"{x:,}")

Number of Markets (tickers) by Series:


series
KXMLBHIT       664,739
KXMLBHR      1,577,108
KXMLBHRR       476,150
KXMLBKS        917,791
KXMLBOUTS       20,652
KXMLBRBI        19,276
KXMLBSB          6,654
KXMLBTB        373,124
Name: ticker, dtype: object

In [112]:
print('First and Last Trade Dates by Series:')
final.groupby('series').agg(
    first_trade_date = ('created_time', 'min'),
    last_trade_date  = ('created_time', 'max')
)

First and Last Trade Dates by Series:


,first_trade_date,last_trade_date
series,,
KXMLBHIT,2026-03-24 01:12:10.540550-04:00,2026-07-11 01:55:17.637658-04:00
KXMLBHR,2026-03-23 23:10:16.259495-04:00,2026-07-11 01:54:22.081233-04:00
KXMLBHRR,2026-03-24 01:20:24.198216-04:00,2026-07-11 01:26:44.142407-04:00
KXMLBKS,2026-03-23 21:36:06.177683-04:00,2026-07-11 01:47:34.917430-04:00
KXMLBOUTS,2026-06-22 21:20:48.669176-04:00,2026-07-11 00:22:18.983520-04:00
KXMLBRBI,2026-06-22 19:42:19.477986-04:00,2026-07-11 01:54:34.965661-04:00
KXMLBSB,2026-06-22 19:30:28.382288-04:00,2026-07-11 01:53:43.843023-04:00
KXMLBTB,2026-03-24 18:48:12.700517-04:00,2026-07-11 01:24:09.003297-04:00


# Analysis

In [113]:
LIQUIDITY_AMOUNT: float = 25.0      # Pre-game liquidity size for simulation
TRADES: int = 5                     # Required number of pre-game trades for simulation
DOLLAR_VOLUME: float = 50.0         # Required dollar volume pre-game for simulation
KALSHI_MAKER_FEE: float = 0.0175    # Kalshi maker fee for June 2026

### Analysis - **Home Run** in a MLB Game

In [114]:
# Add number of home runs column

homeruns_df['n_homeruns'] = homeruns_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

homerun_frequencies = Counter(homeruns_df['n_homeruns'])

homerun_frequencies

Counter({1: 1457788, 2: 115871, 3: 3449})

In [115]:
# Total dollar volume by n home runs market

print("Total dollar volume by n home runs market:")
homeruns_df.groupby('n_homeruns')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n home runs market:


n_homeruns
1    $26,554,644
2     $1,443,994
3        $71,488
Name: dollar_amt, dtype: object

In [116]:
# Aggregate probabilities

results = []

for homeruns, freq in sorted(homerun_frequencies.items()):
    df = homeruns_df[homeruns_df['n_homeruns'] == homeruns].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 30, TRADES, DOLLAR_VOLUME) # Large number of observed markets

    results.append({
        'homeruns'       : f"{homeruns}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['homeruns'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['homeruns'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['homeruns'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Homeruns Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Home Run Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [117]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['homeruns']} homeruns ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ homeruns ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Luis García,38.0,1059.0,22094.61,0.29,0.14,0.14,0.15,0.15
Munetaka Murakami,53.0,6051.0,163384.86,0.36,0.22,0.23,0.14,0.13
Owen Caissie,34.0,751.0,21871.35,0.24,0.12,0.11,0.11,0.13
Angel Martínez,34.0,786.0,20113.95,0.21,0.10,0.11,0.11,0.10
Ben Rice,83.0,9655.0,181963.79,0.33,0.22,0.22,0.10,0.11
...,...,...,...,...,...,...,...,...
Vladimir Guerrero Jr.,87.0,7662.0,177047.41,0.06,0.15,0.15,-0.09,-0.09
Eugenio Suárez,59.0,1837.0,46244.33,0.10,0.19,0.18,-0.09,-0.08
Logan O'Hoppe,35.0,598.0,18099.02,0.06,0.15,0.14,-0.09,-0.08



──────────── 2+ homeruns ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Juan Soto,43.0,800.0,11246.68,0.05,0.03,0.03,0.02,0.02
Matt Olson,40.0,588.0,6596.92,0.05,0.03,0.03,0.02,0.02
Kyle Schwarber,80.0,2894.0,45355.69,0.05,0.04,0.04,0.01,0.01
Yordan Alvarez,47.0,1191.0,9503.74,0.04,0.04,0.03,0.01,0.01
Nick Kurtz,58.0,1590.0,25618.87,0.03,0.04,0.03,-0.00,0.00
Pete Crow-Armstrong,39.0,742.0,9227.67,0.03,0.03,0.03,-0.00,-0.00
Bryce Harper,47.0,858.0,8134.35,0.00,0.02,0.02,-0.02,-0.02
Julio Rodríguez,48.0,721.0,6405.13,0.00,0.02,0.02,-0.02,-0.02
Aaron Judge,58.0,2745.0,28835.47,0.02,0.04,0.04,-0.03,-0.02



──────────── 3+ homeruns ────────────
No data available for given constraint.


In [118]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every home run prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_homeruns = set(int(r['homeruns'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_homeruns_map = dict()
for n in n_homeruns:
    df = homeruns_df[homeruns_df['n_homeruns'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_homeruns'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_homeruns_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_homeruns_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_homeruns'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Home Run Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Home Runs'
)

fig.show()

In [119]:
# Simulation Statistics: Sell Yes, by Home Run Threshold

stats_rows = []
for n, data in n_homeruns_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_homeruns': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_homeruns')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_homeruns'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_homeruns'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_homeruns'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Home Run Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Home Runs', row=1, col=1)
fig.update_xaxes(title_text='n+ Home Runs', row=1, col=2)
fig.update_xaxes(title_text='n+ Home Runs', row=2, col=1)
fig.update_xaxes(title_text='n+ Home Runs', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Strikeouts** in a MLB Game

In [120]:
# Add number of strikeouts column

strikeouts_df['n_strikeouts'] = strikeouts_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

strikeout_frequencies = Counter(strikeouts_df['n_strikeouts'])

strikeout_frequencies

Counter({6: 210592,
         5: 192170,
         7: 161363,
         4: 108585,
         8: 97012,
         9: 49972,
         3: 39938,
         10: 30092,
         2: 12361,
         11: 10052,
         12: 4350,
         13: 988,
         1: 193,
         14: 123})

In [121]:
# Total dollar volume by n strikeouts market

print("Total dollar volume by n strikeouts market:")
strikeouts_df.groupby('n_strikeouts')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n strikeouts market:


n_strikeouts
1         $9,468
2       $584,749
3     $1,561,691
4     $4,368,139
5     $8,045,593
6     $7,954,114
7     $5,255,547
8     $2,670,843
9     $1,081,199
10      $565,369
11      $183,988
12       $67,107
13       $17,192
14        $1,232
Name: dollar_amt, dtype: object

In [122]:
# Aggregate probabilities

results = []

for strikeouts, freq in sorted(strikeout_frequencies.items()):
    df = strikeouts_df[strikeouts_df['n_strikeouts'] == strikeouts].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 5, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'strikeouts'   : f"{strikeouts}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['strikeouts'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['strikeouts'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['strikeouts'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Strikeouts Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Strikeout Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [123]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['strikeouts']} strikeouts ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 2+ strikeouts ────────────
No data available for given constraint.

──────────── 3+ strikeouts ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Miles Mikolas,6.0,108.0,4159.65,1.00,0.57,0.60,0.43,0.40
Eduardo Rodriguez,6.0,89.0,2734.09,1.00,0.78,0.77,0.22,0.23
Connor Prielipp,7.0,157.0,5172.00,1.00,0.81,0.75,0.19,0.25
Tomoyuki Sugano,12.0,412.0,19707.80,0.75,0.57,0.55,0.18,0.20
Andrew Abbott,7.0,72.0,1142.02,1.00,0.82,0.83,0.18,0.17
J.T. Ginn,6.0,128.0,739.46,1.00,0.82,0.87,0.18,0.13
Casey Mize,7.0,106.0,2492.04,1.00,0.83,0.80,0.17,0.20
Jake Bennett,7.0,95.0,1804.55,0.86,0.69,0.65,0.16,0.21
Jeffrey Springs,6.0,89.0,8693.51,1.00,0.84,0.84,0.16,0.16



──────────── 4+ strikeouts ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Jake Irvin,10.0,294.0,12633.28,0.90,0.57,0.56,0.33,0.34
Robert Gasser,6.0,211.0,10048.98,0.83,0.55,0.56,0.29,0.27
Troy Melton,7.0,337.0,10380.82,0.86,0.60,0.61,0.26,0.25
Brandon Sproat,14.0,545.0,24014.13,0.86,0.61,0.60,0.25,0.26
JR Ritchie,6.0,248.0,12189.89,0.83,0.60,0.58,0.24,0.25
...,...,...,...,...,...,...,...,...
Chris Bassitt,8.0,369.0,15148.09,0.25,0.55,0.54,-0.30,-0.29
Javier Assad,6.0,287.0,15955.63,0.17,0.49,0.47,-0.33,-0.30
Steven Matz,7.0,153.0,4480.27,0.29,0.64,0.61,-0.35,-0.32



──────────── 5+ strikeouts ────────────


...


──────────── 12+ strikeouts ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Dylan Cease,7.0,156.0,1307.55,0.14,0.09,0.09,0.05,0.05
Jacob Misiorowski,12.0,682.0,9833.27,0.17,0.15,0.13,0.01,0.04
Chase Burns,6.0,122.0,938.90,0.00,0.06,0.06,-0.06,-0.06
Shohei Ohtani,6.0,137.0,1348.73,0.00,0.06,0.06,-0.06,-0.06



──────────── 13+ strikeouts ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Jacob Misiorowski,8.0,417.0,6285.42,0.12,0.09,0.09,0.03,0.04



──────────── 14+ strikeouts ────────────
No data available for given constraint.


In [124]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every strikeout prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_strikeouts = set(int(r['strikeouts'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_strikeouts_map = dict()
for n in n_strikeouts:
    df = strikeouts_df[strikeouts_df['n_strikeouts'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_strikeouts'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_strikeouts_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_strikeouts_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_strikeouts'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Strikeout Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Strikeouts'
)

fig.show()

In [125]:
# Simulation Statistics: Sell Yes, by Strikeout Threshold

stats_rows = []
for n, data in n_strikeouts_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_strikeouts': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_strikeouts')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_strikeouts'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_strikeouts'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_strikeouts'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Strikeout Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Strikeouts', row=1, col=1)
fig.update_xaxes(title_text='n+ Strikeouts', row=1, col=2)
fig.update_xaxes(title_text='n+ Strikeouts', row=2, col=1)
fig.update_xaxes(title_text='n+ Strikeouts', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Hits** in a MLB Game

In [126]:
# Add number of hits column

hits_df['n_hits'] = hits_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

hit_frequencies = Counter(hits_df['n_hits'])

hit_frequencies

Counter({1: 369754, 2: 210556, 3: 77148, 4: 7281})

In [127]:
# Total dollar volume by n hits market

print("Total dollar volume by n hits market:")
hits_df.groupby('n_hits')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n hits market:


n_hits
1    $9,458,807
2    $3,720,019
3      $959,992
4       $62,145
Name: dollar_amt, dtype: object

In [128]:
# Aggregate probabilities

results = []

for hits, freq in sorted(hit_frequencies.items()):
    df = hits_df[hits_df['n_hits'] == hits].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 30, TRADES, DOLLAR_VOLUME) # Large number of observed markets

    results.append({
        'hits'         : f"{hits}+",
        'pregame_prob' : pregame_prob * 100,
        'hit_rate'     : hit_rate * 100,
        'n'            : n,
        'players'      : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['hits'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['hits'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['hits'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Hits Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Hit Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [129]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['hits']} hits ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ hits ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Jonathan Aranda,36.0,447.0,16298.48,0.78,0.65,0.65,0.13,0.13
Oneil Cruz,38.0,573.0,13496.78,0.74,0.63,0.64,0.11,0.10
Pete Alonso,55.0,789.0,41889.49,0.69,0.60,0.65,0.10,0.04
Freddie Freeman,90.0,2614.0,115652.89,0.73,0.65,0.69,0.09,0.04
Elly De La Cruz,60.0,883.0,23947.85,0.73,0.65,0.65,0.08,0.08
Matt Olson,85.0,1837.0,59769.60,0.73,0.65,0.65,0.08,0.08
Kevin McGonigle,40.0,476.0,18822.13,0.75,0.67,0.67,0.08,0.08
Pete Crow-Armstrong,82.0,2157.0,90583.51,0.73,0.66,0.66,0.07,0.07
Juan Soto,67.0,1203.0,33704.94,0.73,0.66,0.66,0.07,0.07



──────────── 2+ hits ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Jung Hoo Lee,36.0,632.0,15477.46,0.42,0.32,0.31,0.10,0.11
Elly De La Cruz,37.0,463.0,7204.60,0.35,0.28,0.26,0.07,0.09
Ketel Marte,31.0,585.0,23651.25,0.32,0.27,0.29,0.06,0.03
Freddie Freeman,87.0,2441.0,45853.15,0.34,0.29,0.30,0.05,0.04
Juan Soto,45.0,720.0,11043.11,0.31,0.27,0.26,0.04,0.05
Gunnar Henderson,34.0,456.0,14440.54,0.26,0.24,0.28,0.03,-0.02
Luis Arraez,43.0,643.0,12729.92,0.37,0.34,0.33,0.03,0.04
Kyle Schwarber,55.0,751.0,13786.00,0.25,0.22,0.22,0.03,0.03
Ronald Acuña Jr.,37.0,667.0,9610.84,0.32,0.30,0.30,0.02,0.02



──────────── 3+ hits ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Shohei Ohtani,61.0,1158.0,12695.98,0.1,0.09,0.09,0.01,0.01



──────────── 4+ hits ────────────
No data available for given constraint.


In [130]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every hits prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_hits = set(int(r['hits'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_hits_map = dict()
for n in n_hits:
    df = hits_df[hits_df['n_hits'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_hits'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_hits_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_hits_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_hits'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Hit Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Hits'
)

fig.show()

In [131]:
# Simulation Statistics: Sell Yes, by Hit Threshold

stats_rows = []
for n, data in n_hits_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_hits': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_hits')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_hits'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_hits'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_hits'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Hit Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Hits', row=1, col=1)
fig.update_xaxes(title_text='n+ Hits', row=1, col=2)
fig.update_xaxes(title_text='n+ Hits', row=2, col=1)
fig.update_xaxes(title_text='n+ Hits', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Hits + Runs + RBIs** in a MLB Game

In [132]:
# Add number of hits + runs + rbis column

hits_runs_rbi_df['n_hits_runs_rbi'] = hits_runs_rbi_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

hits_runs_rbi_frequencies = Counter(hits_runs_rbi_df['n_hits_runs_rbi'])

hits_runs_rbi_frequencies

Counter({2: 217610, 1: 107245, 3: 85513, 4: 41299, 5: 24483})

In [133]:
# Total dollar volume by n hits + runs + rbis market

print("Total dollar volume by n hits + runs + rbi market:")
hits_runs_rbi_df.groupby('n_hits_runs_rbi')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n hits + runs + rbi market:


n_hits_runs_rbi
1    $2,228,630
2    $5,498,724
3    $1,622,438
4      $539,262
5      $295,367
Name: dollar_amt, dtype: object

In [134]:
# Aggregate probabilities

results = []

for hits_runs_rbi, freq in sorted(hits_runs_rbi_frequencies.items()):
    df = hits_runs_rbi_df[hits_runs_rbi_df['n_hits_runs_rbi'] == hits_runs_rbi].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 30, TRADES, DOLLAR_VOLUME) # Large number of observed markets

    results.append({
        'hits_runs_rbi' : f"{hits_runs_rbi}+",
        'pregame_prob'  : pregame_prob * 100,
        'hit_rate'      : hit_rate * 100,
        'n'             : n,
        'players'       : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['hits_runs_rbi'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['hits_runs_rbi'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['hits_runs_rbi'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Hits + Runs + RBIs Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Hits + Runs + RBI Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [135]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['hits_runs_rbi']} hits + runs + rbi ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ hits + runs + rbi ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Pete Crow-Armstrong,34.0,510.0,22552.75,0.79,0.71,0.73,0.09,0.06
Bryce Harper,36.0,400.0,19529.28,0.81,0.77,0.76,0.04,0.05
Aaron Judge,36.0,441.0,21220.25,0.75,0.76,0.76,-0.01,-0.01
Vladimir Guerrero Jr.,35.0,391.0,14979.20,0.74,0.77,0.76,-0.03,-0.02
Shohei Ohtani,59.0,733.0,35645.21,0.76,0.80,0.80,-0.04,-0.04
Yordan Alvarez,46.0,505.0,30702.26,0.65,0.78,0.77,-0.12,-0.12



──────────── 2+ hits + runs + rbi ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Otto Lopez,33.0,586.0,18462.43,0.73,0.56,0.57,0.17,0.16
Alec Burleson,31.0,409.0,13329.40,0.71,0.54,0.55,0.17,0.16
Jung Hoo Lee,31.0,660.0,19552.82,0.68,0.52,0.52,0.15,0.16
Jordan Walker,44.0,874.0,27336.50,0.66,0.55,0.54,0.11,0.12
Brice Turang,52.0,907.0,31573.27,0.65,0.54,0.55,0.11,0.10
...,...,...,...,...,...,...,...,...
Cal Raleigh,57.0,1134.0,29732.86,0.35,0.49,0.49,-0.14,-0.14
Austin Riley,31.0,403.0,7605.09,0.35,0.51,0.49,-0.16,-0.14
Jo Adell,44.0,695.0,20652.49,0.36,0.52,0.49,-0.16,-0.13



──────────── 3+ hits + runs + rbi ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
James Wood,42.0,585.0,16140.72,0.57,0.40,0.41,0.17,0.16
Ketel Marte,37.0,532.0,13024.66,0.51,0.43,0.42,0.08,0.09
Pete Crow-Armstrong,37.0,913.0,30228.33,0.49,0.42,0.41,0.07,0.08
Junior Caminero,34.0,522.0,10554.07,0.50,0.43,0.42,0.07,0.08
Freddie Freeman,55.0,1012.0,27410.98,0.45,0.42,0.41,0.03,0.04
Kyle Schwarber,48.0,622.0,17845.15,0.46,0.43,0.41,0.03,0.05
Matt Olson,59.0,946.0,23932.22,0.42,0.40,0.38,0.02,0.04
Yordan Alvarez,59.0,961.0,25651.53,0.46,0.44,0.44,0.01,0.02
Shohei Ohtani,83.0,2730.0,95859.39,0.48,0.49,0.48,-0.00,0.00



──────────── 4+ hits + runs + rbi ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Shohei Ohtani,48.0,747.0,21293.42,0.29,0.34,0.33,-0.04,-0.04



──────────── 5+ hits + runs + rbi ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Shohei Ohtani,43.0,593.0,12474.1,0.21,0.23,0.2,-0.02,0.01


In [136]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every hits + runs + rbis prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_hits_runs_rbi = set(int(r['hits_runs_rbi'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_hits_runs_rbi_map = dict()
for n in n_hits_runs_rbi:
    df = hits_runs_rbi_df[hits_runs_rbi_df['n_hits_runs_rbi'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_hits_runs_rbi'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_hits_runs_rbi_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_hits_runs_rbi_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_hits_runs_rbi'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Hits + Runs + RBI Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Hits + Runs + RBI'
)

fig.show()

In [137]:
# Simulation Statistics: Sell Yes, by Hits + Runs + RBIs Threshold

stats_rows = []
for n, data in n_hits_runs_rbi_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_hits_runs_rbi': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_hits_runs_rbi')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_hits_runs_rbi'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_hits_runs_rbi'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_hits_runs_rbi'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Hits + Runs + RBI Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Hits + Runs + RBI', row=1, col=1)
fig.update_xaxes(title_text='n+ Hits + Runs + RBI', row=1, col=2)
fig.update_xaxes(title_text='n+ Hits + Runs + RBI', row=2, col=1)
fig.update_xaxes(title_text='n+ Hits + Runs + RBI', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Total Bases** in a MLB Game

In [138]:
# Add number of total bases column

total_bases_df['n_total_bases'] = total_bases_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

total_base_frequencies = Counter(total_bases_df['n_total_bases'])

total_base_frequencies

Counter({2: 221816, 3: 74616, 4: 48549, 5: 20994, 6: 6345, 7: 675, 8: 129})

In [139]:
# Total dollar volume by n total bases market

print("Total dollar volume by n total bases market:")
total_bases_df.groupby('n_total_bases')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n total bases market:


n_total_bases
2    $4,622,715
3    $1,094,068
4      $685,304
5      $244,483
6       $62,672
7        $5,918
8          $737
Name: dollar_amt, dtype: object

In [140]:
# Aggregate probabilities

results = []

for total_bases, freq in sorted(total_base_frequencies.items()):
    df = total_bases_df[total_bases_df['n_total_bases'] == total_bases].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 20, TRADES, DOLLAR_VOLUME) # Medium number of observed markets

    results.append({
        'total_bases'   : f"{total_bases}+",
        'pregame_prob'  : pregame_prob * 100,
        'hit_rate'      : hit_rate * 100,
        'n'             : n,
        'players'       : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['total_bases'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['total_bases'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['total_bases'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Total Bases Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Total Bases Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [141]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['total_bases']} total bases ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 2+ total bases ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Brice Turang,37.0,444.0,14025.88,0.62,0.46,0.43,0.17,0.19
Jordan Walker,33.0,517.0,12788.54,0.61,0.46,0.46,0.14,0.15
Ketel Marte,56.0,1833.0,47873.38,0.61,0.48,0.47,0.13,0.14
Oneil Cruz,43.0,844.0,15465.94,0.53,0.44,0.45,0.10,0.08
Matt Olson,90.0,2517.0,66697.11,0.52,0.45,0.44,0.07,0.08
...,...,...,...,...,...,...,...,...
Kevin McGonigle,21.0,245.0,5348.60,0.29,0.43,0.44,-0.15,-0.15
Michael Busch,25.0,344.0,9130.48,0.28,0.44,0.44,-0.16,-0.16
Jo Adell,29.0,360.0,6623.11,0.24,0.41,0.41,-0.17,-0.17



──────────── 3+ total bases ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Pete Crow-Armstrong,29.0,411.0,11001.19,0.45,0.34,0.33,0.11,0.12
Kyle Schwarber,21.0,288.0,8605.91,0.38,0.36,0.33,0.02,0.05
Freddie Freeman,40.0,543.0,7832.05,0.30,0.29,0.28,0.01,0.02
Aaron Judge,32.0,493.0,8985.58,0.34,0.36,0.36,-0.02,-0.02
Bobby Witt Jr.,23.0,471.0,7902.60,0.30,0.33,0.30,-0.03,0.00
Shohei Ohtani,65.0,1400.0,33481.10,0.37,0.41,0.39,-0.04,-0.02
Nick Kurtz,25.0,456.0,7252.15,0.28,0.35,0.32,-0.07,-0.04
Yordan Alvarez,32.0,485.0,12017.95,0.28,0.35,0.33,-0.07,-0.05
Fernando Tatis Jr.,24.0,352.0,8074.36,0.08,0.31,0.30,-0.23,-0.22



──────────── 4+ total bases ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Shohei Ohtani,50.0,736.0,15025.08,0.34,0.32,0.31,0.02,0.03
Yordan Alvarez,23.0,243.0,4670.30,0.30,0.30,0.29,0.00,0.01
Kyle Schwarber,24.0,240.0,6638.54,0.29,0.33,0.29,-0.04,0.00
Fernando Tatis Jr.,29.0,281.0,4860.78,0.10,0.23,0.20,-0.13,-0.10



──────────── 5+ total bases ────────────
No data available for given constraint.

──────────── 6+ total bases ────────────
No data available for given constraint.

──────────── 7+ total bases ────────────
No data available for given constraint.

──────────── 8+ total bases ────────────
No data available for given constraint.


In [142]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every total bases prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_total_bases = set(int(r['total_bases'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_total_bases_map = dict()
for n in n_total_bases:
    df = total_bases_df[total_bases_df['n_total_bases'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_total_bases'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_total_bases_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_total_bases_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_total_bases'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Total Bases Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Total Bases'
)

fig.show()

In [143]:
# Simulation Statistics: Sell Yes, by Total Bases Threshold

stats_rows = []
for n, data in n_total_bases_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_total_bases': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_total_bases')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_total_bases'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_total_bases'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_total_bases'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Total Bases Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Total Bases', row=1, col=1)
fig.update_xaxes(title_text='n+ Total Bases', row=1, col=2)
fig.update_xaxes(title_text='n+ Total Bases', row=2, col=1)
fig.update_xaxes(title_text='n+ Total Bases', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Outs** in a MLB Game

In [144]:
# Add number of outs column

outs_df['n_outs'] = outs_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

out_frequencies = Counter(outs_df['n_outs'])

out_frequencies

Counter({18: 8915,
         16: 3879,
         17: 3758,
         19: 2264,
         15: 1561,
         20: 166,
         14: 87,
         12: 20,
         9: 2})

In [145]:
# Total dollar volume by n outs market

print("Total dollar volume by n outs market:")
outs_df.groupby('n_outs')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n outs market:


n_outs
9         $173
12        $934
14      $3,809
15     $53,283
16    $117,494
17     $82,610
18    $233,126
19     $45,877
20      $5,390
Name: dollar_amt, dtype: object

In [146]:
# Aggregate probabilities

results = []

for outs, freq in sorted(out_frequencies.items()):
    df = outs_df[outs_df['n_outs'] == outs].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 5, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'outs'          : f"{outs}+",
        'pregame_prob'  : pregame_prob * 100,
        'hit_rate'      : hit_rate * 100,
        'n'             : n,
        'players'       : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['outs'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['outs'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['outs'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Outs Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Outs Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [147]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['outs']} outs ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 14+ outs ────────────
No data available for given constraint.

──────────── 15+ outs ────────────
No data available for given constraint.

──────────── 16+ outs ────────────
No data available for given constraint.

──────────── 17+ outs ────────────
No data available for given constraint.

──────────── 18+ outs ────────────
No data available for given constraint.

──────────── 19+ outs ────────────
No data available for given constraint.

──────────── 20+ outs ────────────
No data available for given constraint.


In [148]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every out prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_outs = set(int(r['outs'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_outs_map = dict()
for n in n_outs:
    df = outs_df[outs_df['n_outs'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                           # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,                # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                       # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME          # Strictly more than y dollar volume pre-game
    )
    sim_results['n_outs'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_outs_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_outs_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_outs'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Out Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ Outs'
)

fig.show()

In [149]:
# Simulation Statistics: Buy No, by Outs Threshold

stats_rows = []
for n, data in n_outs_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_outs': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_outs')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_outs'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_outs'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_outs'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Out Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Outs', row=1, col=1)
fig.update_xaxes(title_text='n+ Outs', row=1, col=2)
fig.update_xaxes(title_text='n+ Outs', row=2, col=1)
fig.update_xaxes(title_text='n+ Outs', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

In [150]:
SERIES_TICKERS

{'homeruns': 'KXMLBHR',
 'strikeouts': 'KXMLBKS',
 'hits': 'KXMLBHIT',
 'hits_runs_rbi': 'KXMLBHRR',
 'total_bases': 'KXMLBTB',
 'outs': 'KXMLBOUTS',
 'rbi': 'KXMLBRBI',
 'stolen_bases': 'KXMLBSB'}

### Analysis - **RBIs** in a MLB Game

In [151]:
# Add number of rbis column

rbi_df['n_rbis'] = rbi_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

rbi_frequencies = Counter(rbi_df['n_rbis'])

rbi_frequencies

Counter({1: 13330, 2: 5322, 3: 624})

In [152]:
# Total dollar volume by n rbis market

print("Total dollar volume by n rbis market:")
rbi_df.groupby('n_rbis')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n rbis market:


n_rbis
1    $96,856
2    $30,996
3     $3,819
Name: dollar_amt, dtype: object

In [153]:
# Aggregate probabilities

results = []

for rbis, freq in sorted(rbi_frequencies.items()):
    df = rbi_df[rbi_df['n_rbis'] == rbis].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 5, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'rbis'          : f"{rbis}+",
        'pregame_prob'  : pregame_prob * 100,
        'hit_rate'      : hit_rate * 100,
        'n'             : n,
        'players'       : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['rbis'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['rbis'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['rbis'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>RBI Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='RBI Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [154]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['rbis']} rbis ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ rbis ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Shohei Ohtani,8.0,111.0,2588.83,0.62,0.44,0.44,0.18,0.18
Matt Olson,6.0,63.0,1318.28,0.33,0.36,0.37,-0.03,-0.04
Freddie Freeman,9.0,110.0,4680.60,0.22,0.42,0.40,-0.20,-0.18



──────────── 2+ rbis ────────────
No data available for given constraint.

──────────── 3+ rbis ────────────
No data available for given constraint.


In [155]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every rbi prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_rbis = set(int(r['rbis'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_rbis_map = dict()
for n in n_rbis:
    df = rbi_df[rbi_df['n_rbis'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                      # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,           # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                  # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME     # Strictly more than y dollar volume pre-game
    )
    sim_results['n_rbis'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_rbis_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_rbis_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_rbis'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>RBI Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+ RBIs'
)

fig.show()

In [156]:
# Simulation Statistics: Buy No, by RBIs Threshold

stats_rows = []
for n, data in n_rbis_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_rbis': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_rbis')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_rbis'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_rbis'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_rbis'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by RBI Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ RBIs', row=1, col=1)
fig.update_xaxes(title_text='n+ RBIs', row=1, col=2)
fig.update_xaxes(title_text='n+ RBIs', row=2, col=1)
fig.update_xaxes(title_text='n+ RBIs', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()

### Analysis - **Stolen Bases** in a MLB Game

In [157]:
# Add number of stolen bases column

stolen_bases_df['n_stolen_bases'] = stolen_bases_df['ticker'].apply(lambda x: x.split('-')[-1]).astype(int)

stolen_base_frequencies = Counter(stolen_bases_df['n_stolen_bases'])

stolen_base_frequencies

Counter({1: 6654})

In [158]:
# Total dollar volume by n stolen bases market

print("Total dollar volume by n stolen bases market:")
stolen_bases_df.groupby('n_stolen_bases')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Total dollar volume by n stolen bases market:


n_stolen_bases
1    $126,581
Name: dollar_amt, dtype: object

In [159]:
# Aggregate probabilities

results = []

for stolen_bases, freq in sorted(stolen_base_frequencies.items()):
    df = stolen_bases_df[stolen_bases_df['n_stolen_bases'] == stolen_bases].copy()

    pregame_prob, hit_rate, n = agg_probabilities(df, TRADES, DOLLAR_VOLUME)

    if n is None:
        continue

    players = player_hit_rates(df, 5, TRADES, DOLLAR_VOLUME) # Small number of observed markets

    results.append({
        'stolen_bases'  : f"{stolen_bases}+",
        'pregame_prob'  : pregame_prob * 100,
        'hit_rate'      : hit_rate * 100,
        'n'             : n,
        'players'       : players
    })

results_df = pd.DataFrame(results)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['stolen_bases'],
    y=results_df['pregame_prob'],
    name='Pre-Game',
    customdata=results_df['n'],
    hovertemplate='Pre-Game Prob: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=results_df['stolen_bases'],
    y=results_df['hit_rate'],
    name='Hit Rate',
    customdata=results_df['n'],
    hovertemplate='Hit Rate: %{y:.2f}%<br>Markets: %{customdata}<extra></extra>'
))

max_height = results_df[['pregame_prob', 'hit_rate']].max(axis=1)

for x, n, height in zip(results_df['stolen_bases'], results_df['n'], max_height):
    fig.add_annotation(
        x=x,
        y=height + 3,
        text=f"n={n:,.0f}",
        showarrow=False,
        font={'size': 11, 'color': '#b0b0b0'}
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    title={
        'text': (
            "<b>Pre-Game Market Probability vs. Observed Hit Rate</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Stolen Base Prop Markets, by Threshold | 'n' trade count</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Stolen Base Threshold',
    yaxis_title='Probability (%)',
    yaxis_range=[0, max_height.max() + 15],
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.show()

In [160]:
# Player hit rates

for row in results:
    print(f"\n──────────── {row['stolen_bases']} stolen bases ────────────")
    if row['players'] is not None:
        display(row['players'].sort_values('hit_rate_dvwa_delta', ascending=False))
    else:
        print('No data available for given constraint.')


──────────── 1+ stolen bases ────────────


,market_count,pregame_trade_count,dollar_volume,hit_rate,dvwa_pregame_yes_px,median_pregame_yes_px,hit_rate_dvwa_delta,hit_rate_median_delta
player,,,,,,,,
Pete Crow-Armstrong,11.0,242.0,2239.63,0.18,0.24,0.22,-0.06,-0.04
Bobby Witt Jr.,8.0,88.0,955.28,0.12,0.24,0.23,-0.11,-0.11


In [161]:
# Simulation: Sell Yes Limit Order
#   - A market maker provides $25 worth of liquidity on every stolen base prop market
#   - Fill price is naively approximated as the historical dollar-volume-weighted average pre-game price
#   - The wealth path evolves over time, starting at zero

n_stolen_bases = set(int(r['stolen_bases'][:-1]) for r in results if r['n'] > 500) # More than 500 trades

n_stolen_bases_map = dict()
for n in n_stolen_bases:
    df = stolen_bases_df[stolen_bases_df['n_stolen_bases'] == n].copy()
    sim_results = simulation(
        df=df,
        side='no',                           # Selling 'yes' is the same as buying 'no'
        liquidity_amt=LIQUIDITY_AMOUNT,
        fee=KALSHI_MAKER_FEE,                # Assume the 'yes' contract is sold by a market maker offering liquidity to a retail taker
        by='dvwa',
        trades=TRADES,                       # Strictly more than x trades occured pre-game
        dollar_volume=DOLLAR_VOLUME          # Strictly more than y dollar volume pre-game
    )
    sim_results['n_stolen_bases'] = n 

    w = simulation_statistics(
        df=sim_results,
        group_returns_by='weekly'
    )     
    m = simulation_statistics(
        df=sim_results,
        group_returns_by='monthly'
    )  

    n_stolen_bases_map[n] = {
        'simulation_results'       : sim_results,
        'weekly_simulation_stats'  : w,
        'monthly_simulation_stats' : m,
    }

combined_df = pd.concat(
    [v['simulation_results'] for v in n_stolen_bases_map.values()],
    ignore_index=True
)

fig = px.line(
    combined_df,
    x='close_time',
    y='cum_net_winnings',
    color='n_stolen_bases'        
)

fig.update_layout(
    template='plotly_dark',
    title={
        'text': (
            f"<b>Simulated Wealth Path of a Market Maker Selling ${LIQUIDITY_AMOUNT} Worth of 'Yes' Contracts Pre-Game</b><br>"
            f"<span style='font-size: 15px; color: #b0b0b0;'>Stolen Bases Prop Simulation | Kalshi Fees Considered</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Date',
    yaxis_title='Cumulative Net Winnings ($)',
    hovermode='x unified',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text='n+Stolen Bases'
)

fig.show()

In [162]:
# Simulation Statistics: Buy No, by Stolen Base Threshold

stats_rows = []
for n, data in n_stolen_bases_map.items():
    for period, stats in [
        ('weekly', data['weekly_simulation_stats']),
        ('monthly', data['monthly_simulation_stats']),
    ]:
        stats_rows.append({
            'n_stolen_bases': f"{n}+",
            'period': period,
            'mean': stats['mean'],
            'median': stats['median'],
            'std': stats['std'],
            'mean_std_ratio': stats['mean'] / stats['std'] if stats['std'] else None,
            'count': stats['count'],
        })

stats_df = pd.DataFrame(stats_rows).sort_values('n_stolen_bases')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Weekly Mean vs. Median',
        'Monthly Mean vs. Median',
        'Weekly Mean / Std',
        'Monthly Mean / Std',
    ),
)

# --- Top row: mean vs. median returns, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_stolen_bases'],
            y=sub['mean'],
            name='Mean Return',
            marker_color='#636efa',
            customdata=sub['count'],
            hovertemplate='Mean: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

    fig.add_trace(
        go.Bar(
            x=sub['n_stolen_bases'],
            y=sub['median'],
            name='Median Return',
            marker_color='#00cc96',
            customdata=sub['count'],
            hovertemplate='Median: %{y:.2%}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='median_return',
            showlegend=(col == 1),
        ),
        row=1, col=col
    )

# --- Bottom row: mean/std ratio, weekly (col 1) and monthly (col 2) ---
for col, period in zip([1, 2], ['weekly', 'monthly']):
    sub = stats_df[stats_df['period'] == period]

    fig.add_trace(
        go.Bar(
            x=sub['n_stolen_bases'],
            y=sub['mean_std_ratio'],
            name='Mean / Std',
            marker_color='#ffa15a',
            customdata=sub['count'],
            hovertemplate='Mean/Std: %{y:.2f}<br>n periods: %{customdata}<extra></extra>',
            legendgroup='mean_std',
            showlegend=(col == 1),
        ),
        row=2, col=col
    )

fig.update_layout(
    template='plotly_dark',
    barmode='group',
    height=800,
    title={
        'text': (
            "<b>Weekly vs. Monthly Return Statistics by Stolen Base Threshold</b><br>"
            "<span style='font-size: 15px; color: #b0b0b0;'>Sell 'Yes' | Return = Net Winnings / Capital at Risk</span>"
        ),
        'font': {'size': 20, 'color': '#ffffff'},
        'x': 0.5,
        'xanchor': 'center'
    },
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    legend_title_text=''
)

fig.update_xaxes(title_text='n+ Stolen Bases', row=1, col=1)
fig.update_xaxes(title_text='n+ Stolen Bases', row=1, col=2)
fig.update_xaxes(title_text='n+ Stolen Bases', row=2, col=1)
fig.update_xaxes(title_text='n+ Stolen Bases', row=2, col=2)

fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Return', tickformat='.0%', row=1, col=2)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=1)
fig.update_yaxes(title_text='Mean / Std (ratio)', row=2, col=2)

fig.show()